# GARCH in Neural Network form (2 hidden layers)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the provided Gogoro data
gogoro_df = pd.read_csv('/content/Gogoro Stock Price History.csv')

# Sort by date to ensure proper time series split
gogoro_df['Date'] = pd.to_datetime(gogoro_df['Date'])
gogoro_df = gogoro_df.sort_values(by='Date')

# Split the data into training and evaluation sets (e.g., 80% train, 20% test)
train_size = int(len(gogoro_df) * 0.8)
train_df = gogoro_df.iloc[:train_size]
eval_df = gogoro_df.iloc[train_size:]

# Save the split data to temporary CSV files
train_df.to_csv('gogoro_training.csv', index=False)
eval_df.to_csv('gogoro_evaluation.csv', index=False)

print(f"Created 'gogoro_training.csv' with {len(train_df)} rows.")
print(f"Created 'gogoro_evaluation.csv' with {len(eval_df)} rows.")

Created 'gogoro_training.csv' with 200 rows.
Created 'gogoro_evaluation.csv' with 51 rows.


In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from torchsummary import summary
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

def build_eval_loader(path: str, window: int, batch_size: int = 64):
    # same preprocessing as load_and_process()
    df = pd.read_csv(path, parse_dates=['Date'], index_col='Date')
    df['return'] = df['Price'].pct_change()
    df = df.dropna()
    df['return'] -= df['return'].mean()
    df['hist_var'] = df['return'].rolling(window).var().shift(1)
    df = df.dropna()

    rets = torch.tensor(df['return'].values, dtype=torch.float32)
    vars_ = torch.tensor(df['hist_var'].values, dtype=torch.float32)

    eval_ds = GarchDataset(rets, vars_, window)
    eval_dl = DataLoader(eval_ds, batch_size=batch_size, shuffle=False)
    return eval_dl

def mse_realized_vol(model, dataloader, device='cpu', annualize=False, trading_days=252):
    """MSE between predicted σ̂_t and realized |r_t| on a loader."""
    model.eval()
    total, count = 0.0, 0
    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device); y = y.to(device)       # y is demeaned return (ε_t)
            sigma = model(x)                         # σ̂_t (daily std)
            target = y.abs()                         # realized daily vol proxy |r_t|
            if annualize:
                scale = torch.sqrt(torch.tensor(trading_days, dtype=sigma.dtype, device=device))
                sigma  = sigma  * scale
                target = target * scale
            mse_batch = torch.mean((sigma - target)**2)
            total += mse_batch.item() * x.size(0)
            count += x.size(0)
    return total / count


# 1. Function to load and preprocess a SPY CSV
def load_and_process(path: str, window: int):
    df = pd.read_csv(
        path,
        parse_dates=['Date'],
        index_col='Date'
    )
    # compute returns
    df['return'] = df['Price'].pct_change()
    df = df.dropna()
    df['return'] -= df['return'].mean()
    # compute historical variance (lagged)
    df['hist_var'] = df['return'].rolling(window).var().shift(1)
    df = df.dropna()
    # convert to tensors
    rets = torch.tensor(df['return'].values, dtype=torch.float32)
    vars_ = torch.tensor(df['hist_var'].values, dtype=torch.float32)
    return rets, vars_

# 2. Dataset: windowed past returns + past variances → next ε_t
class GarchDataset(Dataset):
    def __init__(self, returns: torch.Tensor, variances: torch.Tensor, window: int):
        self.window = window
        self.eps = returns
        self.vars = variances
    def __len__(self):
        return len(self.eps) - self.window
    def __getitem__(self, idx):
        r_lags = self.eps[idx: idx + self.window]
        v_lags = self.vars[idx: idx + self.window]
        x = torch.cat([r_lags, v_lags], dim=0)
        y = self.eps[idx + self.window]
        return x, y

# 3. Model: feedforward → positive σ̂_t
class NeuralGARCH(nn.Module):
    def __init__(self, window: int, hidden_sizes=(64, 32)):
        super().__init__()
        inp_dim = 2 * window
        layers = []
        for h in hidden_sizes:
            layers += [nn.Linear(inp_dim, h), nn.ReLU()]
            inp_dim = h
        layers.append(nn.Linear(inp_dim, 1))
        self.net = nn.Sequential(*layers)
        self.softplus = nn.Softplus()
    def forward(self, x):
        raw = self.net(x)
        return self.softplus(raw).squeeze(-1)

# 4. GARCH loss (Gaussian log-likelihood)
def garch_loss(eps: torch.Tensor, sigma: torch.Tensor):
    var = sigma**2
    return torch.mean(eps**2 / var + torch.log(var))

# 5. Training loop
def train(model, dataloader, lr=1e-3, epochs=50, device='cpu'):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.to(device)
    for ep in range(1, epochs+1):
        total_loss = 0.0
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            sigma = model(x)
            loss = garch_loss(y, sigma)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total_loss += loss.item() * x.size(0)
        avg = total_loss / len(dataloader.dataset)
        if ep % 10 == 0:
            print(f"Epoch {ep:03d}  Train Loss {avg:.6f}")
    return model

# if __name__ == '__main__':
#     window = 20  # your ARCH/GARCH lag length

#     # Load train and verification data
#     rets_train, vars_train = load_and_process('SPY training.csv', window)
#     rets_val,   vars_val   = load_and_process('SPY verification.csv', window)

#     # Datasets & loaders
#     train_ds = GarchDataset(rets_train, vars_train, window)
#     val_ds   = GarchDataset(rets_val,   vars_val,   window)
#     train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
#     val_dl   = DataLoader(val_ds,   batch_size=64, shuffle=False)

#     # Train
#     model  = NeuralGARCH(window=window, hidden_sizes=(128, 64))
#     device = 'cuda' if torch.cuda.is_available() else 'cpu'
#     trained = train(model, train_dl, lr=1e-3, epochs=100, device=device)
#     MODEL_PATH = Path('garch_nn.pth')
#     torch.save(model.state_dict(), MODEL_PATH)
#     print('Saved weights to:', MODEL_PATH.resolve())

#     # Verify (likelihood loss)
#     model.eval()
#     total_val_loss = 0.0
#     with torch.no_grad():
#         for x, y in val_dl:
#             x, y = x.to(device), y.to(device)
#             sigma = model(x)
#             total_val_loss += garch_loss(y, sigma).item() * x.size(0)
#     avg_val = total_val_loss / len(val_ds)
#     print(f"Verification Loss: {avg_val:.6f}")

#     # --- NEW: Evaluate MSE on separate dataset file ---
#     eval_path = 'SPY ETF Stock Price History 2021 to 2025.csv'
#     eval_dl = build_eval_loader(eval_path, window, batch_size=64)
#     mse_eval = mse_realized_vol(model, eval_dl, device=device, annualize=False)
#     print(f"MSE (σ̂ vs |r|) on '{eval_path}': {mse_eval:.6e}")

#     # Optional: annualized comparison
#     mse_eval_ann = mse_realized_vol(model, eval_dl, device=device, annualize=True)
#     print(f"MSE (annualized σ̂ vs |r|·√252) on '{eval_path}': {mse_eval_ann:.6e}")

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

# assumes NeuralGARCH is already defined and `model` is trained
# and you know `window` and `device`

def _prep_inference_df(path: str, window: int) -> pd.DataFrame:
    """Load file and build lagged features consistent with training preprocessing."""
    df = pd.read_csv(path, parse_dates=['Date'], index_col='Date')
    df['return'] = df['Price'].pct_change()
    df = df.dropna()
    df['return'] = df['return'] - df['return'].mean()
    # one-day-ahead variance uses only past info
    df['hist_var'] = df['return'].rolling(window).var().shift(1)
    df = df.dropna()
    return df

def _build_feature_matrix(df: pd.DataFrame, window: int) -> tuple[torch.Tensor, pd.DatetimeIndex]:
    """Build X with shape (N, 2*window). Each row uses past 20 returns and past 20 variances."""
    rets = df['return'].to_numpy(dtype=np.float32)
    vars_ = df['hist_var'].to_numpy(dtype=np.float32)

    X = []
    idx = []
    # for each t, use [t-window, ..., t-1] to predict σ_t
    for t in range(window, len(rets)):
        r_lags = rets[t-window:t]
        v_lags = vars_[t-window:t]
        X.append(np.concatenate([r_lags, v_lags], axis=0))
        idx.append(df.index[t])  # align prediction with day t
    X = np.stack(X).astype(np.float32)
    return torch.from_numpy(X), pd.DatetimeIndex(idx)

@torch.no_grad()
def run_model_over_file(model, path: str, window: int, device='cpu', batch_size=1024) -> pd.DataFrame:
    """Return a DataFrame with σ̂_t for each day in the file (after warmup)."""
    model.eval()
    df = _prep_inference_df(path, window)
    X, idx = _build_feature_matrix(df, window)

    # batched inference on correct device
    X = X.to(device)
    preds = []
    for i in range(0, X.size(0), batch_size):
        preds.append(model(X[i:i+batch_size]).cpu().numpy())  # σ̂_t (daily std)
    sigma_hat = np.concatenate(preds, axis=0)

    out = pd.DataFrame(index=idx)
    out['sigma_hat_daily'] = sigma_hat
    out['sigma_hat_annual'] = out['sigma_hat_daily'] * np.sqrt(252.0)
    # realized proxies aligned to same days
    r_aligned = df.loc[idx, 'return']
    out['realized_abs_daily'] = r_aligned.abs().values.astype(np.float32)
    out['realized_var_daily'] = (r_aligned.values.astype(np.float32))**2
    return out

# ----------------- example usage -----------------
# if __name__ == '__main__':
#     window = 20
#     device = 'cuda' if torch.cuda.is_available() else 'cpu'

#     # rebuild architecture exactly as trained and load weights if needed
#     # model = NeuralGARCH(window=window, hidden_sizes=(128, 64)).to(device)
#     # model.load_state_dict(torch.load('garch_nn.pth', map_location=device))

#     # if you already have a trained `model` in memory, ensure it’s on device:
#     model = model.to(device)

#     path_eval = 'SPY ETF Stock Price History 2021 to 2025.csv'
#     preds_df = run_model_over_file(model, path_eval, window, device=device, batch_size=2048)

#     # save and quick peek
#     preds_df.to_csv('spy_2021_2025_vol_forecasts.csv')
#     print(preds_df.head())
#     print(preds_df.tail())
#     print("Saved to spy_2021_2025_vol_forecasts.csv")

In [ ]:
import numpy as np
import pandas as pd
import torch

# --- Use the same robust loader you already have for Investing.com-style files ---
def _parse_price_series(df: pd.DataFrame, price_col: str = "Price") -> pd.Series:
    s = (df[price_col].astype(str)
           .str.replace(',', '', regex=False)
           .str.replace('$', '', regex=False)
           .str.replace('−', '-', regex=False)
           .str.strip())
    return pd.to_numeric(s, errors='coerce')

def _prep_df_from_price(path: str, window: int) -> pd.DataFrame:
    df = pd.read_csv(path)
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    df = df.dropna(subset=['Date']).sort_values('Date').set_index('Date')

    df['Price'] = _parse_price_series(df, 'Price')
    df = df.dropna(subset=['Price'])

    # Decimal returns; demean to match training
    df['return'] = df['Price'].pct_change()
    df = df.dropna(subset=['return'])
    df['return'] = df['return'] - df['return'].mean()

    # Rolling variance feature, lagged one day (uses only past info)
    df['hist_var'] = df['return'].rolling(window).var().shift(1)
    df = df.dropna(subset=['hist_var'])

    # Also store r^2 for realized calculations
    df['r2'] = df['return']**2
    return df

@torch.no_grad()
def tenor_matched_20d_forecast(model, path: str, window: int = 20, device='cpu') -> pd.DataFrame:
    """
    For each date t (after warmup), build a 20-step-ahead variance path:
      - Predict σ_{t+1} ... σ_{t+20} using recursive, expectation-based updates
      - Forecasted 20d annualized vol: sqrt(252 * mean_i σ_{t+i}^2)
      - Realized 20d annualized vol:  sqrt(252 * mean_i r_{t+i}^2)
    Returns a DataFrame with both series aligned at t.
    """
    model.eval()
    df = _prep_df_from_price(path, window)
    idx = df.index

    rets  = df['return'].to_numpy(dtype=np.float32)
    r2    = df['r2'].to_numpy(dtype=np.float32)
    hvar  = df['hist_var'].to_numpy(dtype=np.float32)

    out_idx = []
    fcast_ann_20d = []
    realized_ann_20d = []
    one_day_ann = []  # optional: annualized 1D forecast at t

    # loop t from window to len-20 so we have 20 future days available for realized vol
    for t in range(window, len(rets) - 20):
        # build current lag windows (length == window)
        r_lags   = rets[t-window:t].copy()
        v_lags   = hvar[t-window:t].copy()
        r2_win   = (rets[t-window:t]**2).copy()  # to compute expected rolling variance

        sigma2_path = []

        for k in range(20):
            # features x_t: [last 20 returns, last 20 hist_vars]
            x = np.concatenate([r_lags, v_lags]).astype(np.float32)
            x_tensor = torch.from_numpy(x).unsqueeze(0).to(device)

            sigma = model(x_tensor).item()          # daily std forecast
            sigma2 = sigma * sigma                   # daily variance forecast
            sigma2_path.append(sigma2)

            # update windows for next step (expectation updates)
            # returns: append E[r]=0
            r_lags = np.roll(r_lags, -1);  r_lags[-1] = 0.0
            # squared returns window: append E[r^2] = sigma^2
            r2_win = np.roll(r2_win, -1);  r2_win[-1] = sigma2
            # hist_var lags: append expected rolling variance = mean of r^2 window
            v_lags = np.roll(v_lags, -1);  v_lags[-1] = r2_win.mean()

        # tenor-matched 20d forecast (annualized)
        var20_f = float(np.mean(sigma2_path))
        fcast_ann_20d.append(np.sqrt(252.0 * var20_f))

        # realized forward 20d annualized (use actual future r^2)
        var20_real = float(np.mean(r2[t+1:t+21]))
        realized_ann_20d.append(np.sqrt(252.0 * var20_real))

        # also store annualized 1-day forecast at t (optional diagnostic)
        # rebuild x_t once more to get σ_t+1 without the path (or reuse first sigma)
        x0 = np.concatenate([rets[t-window:t], hvar[t-window:t]]).astype(np.float32)
        one_day = model(torch.from_numpy(x0).unsqueeze(0).to(device)).item() * np.sqrt(252.0)
        one_day_ann.append(one_day)

        out_idx.append(idx[t])

    out = pd.DataFrame(index=pd.DatetimeIndex(out_idx))
    out['sigma_hat_annual_1d'] = one_day_ann
    out['forecast_ann_20d']    = fcast_ann_20d
    out['realized_ann_20d']    = realized_ann_20d

    # quick metrics
    mse  = float(np.mean((out['forecast_ann_20d'] - out['realized_ann_20d'])**2))
    rmse = float(np.sqrt(mse))
    corr = float(out['forecast_ann_20d'].corr(out['realized_ann_20d']))
    print(f"MSE (20d annualized): {mse:.6f}")
    print(f"RMSE (vol points):    {rmse:.6f}  (~{rmse*100:.2f}%)")
    print(f"Corr(forecast, realized): {corr:.3f}")

    return out

# ---------- Example usage ----------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = NeuralGARCH(window=20, hidden_sizes=(128, 64)).to(device)
model.load_state_dict(torch.load('garch_nn.pth', map_location=device))
model.eval()
df_20d = tenor_matched_20d_forecast(model, 'SPY ETF Stock Price History 2021 to 2025.csv', window=20, device=device)
df_20d.to_csv('sigma_vs_realized_20d.csv')
print(df_20d.head())


In [ ]:
!pip -q install arch
import numpy as np
import pandas as pd
from arch.univariate import arch_model

# ---------- Robust loaders for Investing.com-style CSV (Date, Price, Open, High, Low, Vol., Change %) ----------
def _parse_price_series(df: pd.DataFrame, price_col: str = "Price") -> pd.Series:
    s = (df[price_col].astype(str)
           .str.replace(',', '', regex=False)
           .str.replace('$', '', regex=False)
           .str.replace('−', '-', regex=False)  # unicode minus
           .str.strip())
    return pd.to_numeric(s, errors='coerce')

def _prep_df_from_price(path: str, window: int = 20) -> pd.DataFrame:
    df = pd.read_csv(path)
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    df = df.dropna(subset=['Date']).sort_values('Date').set_index('Date')
    df['Price'] = _parse_price_series(df, 'Price')
    df = df.dropna(subset=['Price'])

    # decimal returns (0.01 = 1%)
    df['return'] = df['Price'].pct_change()
    df = df.dropna(subset=['return'])

    # demean to match your NN pipeline
    df['return'] = df['return'] - df['return'].mean()

    # Rescale returns to address DataScaleWarning
    df['return'] = df['return'] * 10

    df['r2'] = df['return']**2
    return df

# ---------- GARCH(1,1) baseline: fit on training, forecast on eval (20d tenor matched) ----------
def garch_baseline_20d(train_path: str,
                       eval_path: str,
                       dist: str = 'normal',   # or 't'
                       window: int = 20):
    # Load data
    train_df = _prep_df_from_price(train_path, window)
    eval_df  = _prep_df_from_price(eval_path,  window)

    y_train = train_df['return']
    y_eval  = eval_df['return']

    # Fit vanilla GARCH(1,1) on training (returns already demeaned → mean='Zero')
    am = arch_model(y_train, mean='Zero', vol='GARCH', p=1, q=1, dist=dist)
    res = am.fit(disp='off')

    # Apply fixed params to the evaluation series; produce rolling 20-step forecasts
    am_eval = arch_model(y_eval, mean='Zero', vol='GARCH', p=1, q=1, dist=dist)
    res_eval = am_eval.fix(res.params)            # filter using trained params

    # Multi-step variance forecasts for every date (start after warmup)
    fore = res_eval.forecast(horizon=20, start=window)   # start at index=window for stability
    var_k = fore.variance                                # DataFrame: rows=dates, cols=h.1..h.20

    # Tenor-matched 20d forecast: average future daily variances, then annualize
    # Remember to un-rescale the variance for the final output
    var20_mean = var_k.iloc[:, :20].mean(axis=1) / 100                 # mean of h.1..h.20, un-rescale by dividing by (10^2)
    garch_ann_20d = np.sqrt(252.0 * var20_mean)                  # annualized vol

    # Realized forward 20d annualized vol on the eval set
    # Remember to un-rescale the r2 for the final output
    fwd_var20_real = eval_df['r2'].rolling(window=20).mean().shift(-20) / 100 # at t: mean of t+1..t+20, un-rescale by dividing by (10^2)
    realized_ann_20d = np.sqrt(252.0 * fwd_var20_real)

    # Align and drop NaNs (last ~20 rows have no forward realized)
    out = pd.DataFrame({
        'garch_ann_20d': garch_ann_20d,
        'realized_ann_20d': realized_ann_20d
    }).dropna()

    # Metrics
    mse  = float(((out['garch_ann_20d'] - out['realized_ann_20d'])**2).mean())
    rmse = float(np.sqrt(mse))
    corr = float(out['garch_ann_20d'].corr(out['realized_ann_20d']))

    print(f"[GARCH(1,1) baseline | dist={dist}]")
    print(f"MSE (20d annualized): {mse:.6f}")
    print(f"RMSE (vol points):    {rmse:.6f}  (~{rmse*100:.2f}%)")
    print(f"Corr(forecast, realized): {corr:.3f}")

    return out, res

# ---------------- Example usage ----------------
# device = 'cuda' if torch.cuda.is_available() else 'cpu'
# model = NeuralGARCH(window=20, hidden_sizes=(128, 64)).to(device)
# model.load_state_dict(torch.load('garch_nn.pth', map_location=device))
# model.eval()
# df_20d = tenor_matched_20d_forecast(model, 'SPY ETF Stock Price History 2021 to 2025.csv', window=20, device=device)
# df_20d.to_csv('sigma_vs_realized_20d.csv')
# print(df_20d.head())
out_df, garch_res = garch_baseline_20d(
    train_path='gogoro_training.csv',
    eval_path='gogoro_evaluation.csv',
    dist='normal',   # or 't'
    window=20
)
out_df.to_csv('garch_20d_vs_realized_gogoro.csv')
print(out_df.head())

[GARCH(1,1) baseline | dist=normal]
MSE (20d annualized): 0.077989
RMSE (vol points):    0.279266  (~27.93%)
Corr(forecast, realized): -0.074
            garch_ann_20d  realized_ann_20d
Date                                       
2026-03-04       0.708923          1.026490
2026-03-05       0.836598          0.864280
2026-03-06       0.955333          0.618866
2026-03-09       0.913249          0.522549
2026-03-10       0.848749          0.536673
